# Informed Align-then-Unlearn — Qwen3.5-4B on Colab Pro (A100 80GB)

End-to-end pipeline: causal tracing to identify the most responsible decoder layer for a target, then align-then-unlearn at that layer with layer-scoped gradient unfreezing.

**Runtime:** Colab Pro → *Runtime → Change runtime type → A100 GPU, High-RAM*.

**Pipeline:**
1. Check GPU
2. Mount Drive (persist HF weights across sessions)
3. Clone repo (`shiv` branch) + install deps
4. Download RWKU benchmark
5. Retokenize `positive_*.json` for the Qwen tokenizer
6. (Optional) `wandb login`
7. Causal tracing for `1_Stephen_King`
8. Read the peak layer out of the results
9. Smoke-test training (1 epoch)
10. Full chained align-then-unlearn run

All steps assume the repo lives at `/content/informed-align-unlearn`.

## 1. Verify the GPU

You want to see `A100-SXM4-80GB` (or `A100 80GB PCIe`) in the output. If you see a T4 or V100, switch the runtime.

In [ ]:
!nvidia-smi

## 2. Mount Drive and pin `HF_HOME`

Qwen3.5-4B is ~8GB in bfloat16. Persisting the cache on Drive means the next session doesn't re-download. Skip this cell if you don't care about re-downloading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

## 3. Clone the repo (branch `shiv`) and install dependencies

The `shiv` branch contains: causal tracing, Qwen3.5-4B config, layer-scoped unfreezing, chained training, and tokenizer-variant data loading.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d informed-align-unlearn ]; then
  git clone https://github.com/AyushSehgal/informed-align-unlearn.git
fi
cd informed-align-unlearn
git fetch origin shiv
git checkout shiv
git pull origin shiv
pip install -q -r requirements.txt

## 4. Download the RWKU benchmark

This pulls the benchmark zip from Google Drive (~a few hundred MB) and unpacks it into `data/rwku/benchmark/`.

In [ ]:
%%bash
cd /content/informed-align-unlearn
bash data/rwku/download_rwku_data.sh
echo '---'
ls data/rwku/benchmark/Target | head

## 5. Retokenize `positive_phi.json` for the Qwen tokenizer

The repo ships with `positive_phi.json` (raw text). The Qwen loader expects `positive_qwen.json`, so we copy-and-filter by the Qwen tokenizer's length budget. 1024 tokens is a safe cap for the ATU training loop.

In [ ]:
%%bash
cd /content/informed-align-unlearn
python data/rwku/retokenize_positive.py \
  --tokenizer Qwen/Qwen3.5-4B \
  --variant qwen \
  --max_tokens 1024 \
  --trust_remote_code

## 6. (Optional) `wandb login`

Metrics get logged to the `informed-align-unlearn` project. Skip if you don't want W&B — the run will still work.

In [ ]:
import wandb
wandb.login()

## 7. Causal tracing for `1_Stephen_King`

ROME-style clean / corrupted / restored forward passes, averaged over RWKU `forget_level{1,2,3}.json` prompts. Writes `outputs/causal_trace/1_Stephen_King/results.json`.

Takes ~10–15 min on an A100 for Qwen3.5-4B. To change the target, edit `--target_id` (e.g. `9_Justin_Bieber`).

In [ ]:
%%bash
cd /content/informed-align-unlearn
bash scripts/run_causal_trace.sh \
  --target_id 1_Stephen_King \
  --model Qwen/Qwen3.5-4B \
  --top_k 5 \
  --noise_multiplier 3.0 \
  --local

## 8. Read the peak layer out of `results.json`

The training step uses this layer as the unlearning hook. The Python variable `peak` gets interpolated into the next shell command via Colab's `{var}` syntax.

In [ ]:
import json
from pathlib import Path

results = json.loads(
    Path('/content/informed-align-unlearn/outputs/causal_trace/1_Stephen_King/results.json').read_text()
)
peak = int(results['peak_layer'])
print('peak_layer =', peak)
print('top_k_layers:')
for entry in results['top_k_layers']:
    print(f"  layer {entry['layer']:>3d}  recovery {entry['recovery']:.4f}")

## 9. Smoke test — 1 epoch, skip the pre-eval

Quick sanity check that the full pipeline loads cleanly (model, data, layer hook, trainer) before committing to a long run. Should finish in a couple of minutes on an A100.

In [ ]:
!cd /content/informed-align-unlearn && bash scripts/run_chained_training.sh --chain "1_Stephen_King:{peak}" --overrides "trainer.max_epochs=1 skip_initial_eval=true" --local

## 10. Full chained align-then-unlearn

Single target, single layer — starting point. Once this works, you can extend the chain:

```
--chain "1_Stephen_King:4,2_Confucius:20,3_Elon_Musk:10"
```

Each step's modified model is fed into the next. Eval metrics are prefixed `pre_chain/` and `post_chain/` in the W&B dashboard so you can see USR/GUR/APR move per target.

In [ ]:
!cd /content/informed-align-unlearn && bash scripts/run_chained_training.sh --chain "1_Stephen_King:{peak}" --local

## Colab gotchas

- **Session disconnects after ~12h of idle.** For long chains, keep the tab focused or use Colab Pro's background execution.
- **A100 vs. L4/V100.** If you end up on a smaller GPU, drop `--device_map auto` and force `torch_dtype: "float16"` in `config/pre_trained_llm/qwen3.5-4b.yaml`, or use a smaller model.
- **`flash_attn` import errors.** Qwen3.5-4B is loaded with `attn_implementation="eager"` to avoid flash-attn kernel compile issues on Colab. Don't change this without also installing flash-attn wheels.
- **Drive I/O is slow.** `HF_HOME` on Drive is fine for one-time downloads, but Lightning checkpoints should stay on `/content/` to avoid stalls. The default `save_dir` already does this.
- **Disk fill.** Colab Pro gives ~200GB scratch — plenty for one model + a few checkpoints, tight if you chain many unlearning steps with full checkpointing. Set `task.training_module.checkpoint_interval=-1` to skip intermediate checkpoints.
- **`trust_remote_code`.** The Qwen config sets this to `true` and the causal trace script auto-adds the flag. Don't override unless you know what you're doing.
- **RWKU download flakes.** `gdown` occasionally throws a quota error. Re-run the cell — it's idempotent.